# Elderly Care Risk Prediction - Model V2
## Improved pipeline with LightGBM, interaction features and biomarker data
**Dataset:** LASI Wave 1 + DBS Biomarker  
**Author:** Tushar Sekhri  
**Date:** July 2026

In [16]:
import pyreadstat
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier

# Load data
df, meta = pyreadstat.read_dta('../data/3_LASI_W1_Individual_v4.dta')
print(f"Loaded: {df.shape}")

Loaded: (73396, 2623)


In [17]:
# import sys
# !{sys.executable} -m pip install lightgbm

In [18]:
# All features including V6 additions
adl_cols = ['ht401','ht402','ht403','ht404','ht405','ht406',
            'ht407','ht408','ht409','ht410','ht411','ht412']

base_features = [
    'dm003', 'dm005', 'ht001_a',
    'ht002', 'ht003', 'ht004', 'ht005', 'ht006',
    'ht007', 'ht008', 'ht009', 'ht010', 'ht301',
    'living_arrangements', 'dm006', 'we001',
    'fs601', 'ht014', 'ht015'
] + adl_cols

ml_df = df[base_features].copy()

# Create target variable
ml_df['needs_care'] = (ml_df[adl_cols] == 1).any(axis=1).astype(int)

# Drop ADL rows with missing
ml_df = ml_df.dropna(subset=adl_cols)

# Fill remaining missing with mode
for col in ml_df.columns:
    if ml_df[col].isnull().sum() > 0:
        ml_df[col] = ml_df[col].fillna(ml_df[col].mode()[0])

# Feature engineering
ml_df['disease_count'] = (ml_df[['ht002','ht003','ht004','ht005','ht006',
                                   'ht007','ht008','ht009','ht010']] == 1).sum(axis=1)
ml_df['age_group'] = pd.cut(ml_df['dm005'], bins=[0,60,70,80,120], labels=[0,1,2,3]).astype(int)
ml_df['lives_alone'] = (ml_df['living_arrangements'] == 1).astype(int)
ml_df['poor_health'] = (ml_df['ht001_a'] >= 4).astype(int)

# Interaction features
ml_df['age_alone'] = ml_df['age_group'] * ml_df['lives_alone']
ml_df['disease_poor_health'] = ml_df['disease_count'] * ml_df['poor_health']

print(f"Clean dataset: {ml_df.shape}")
print(f"Target distribution:\n{ml_df['needs_care'].value_counts()}")

Clean dataset: (73027, 38)
Target distribution:
needs_care
0    48816
1    24211
Name: count, dtype: int64


In [19]:
X = ml_df.drop(columns=['needs_care'] + adl_cols)
y = ml_df['needs_care']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train_scaled, y_train)

# Train LightGBM
lgbm_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)
lgbm_model.fit(X_train_sm, y_train_sm)

y_pred_lgbm = lgbm_model.predict(X_test_scaled)
y_pred_proba_lgbm = lgbm_model.predict_proba(X_test_scaled)[:, 1]

print("LightGBM Results:")
print(classification_report(y_test, y_pred_lgbm, target_names=['No Care Needed', 'Care Needed']))
print(f"ROC AUC Score: {roc_auc_score(y_test, y_pred_proba_lgbm):.4f}")

[LightGBM] [Info] Number of positive: 39052, number of negative: 39052
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004110 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2985
[LightGBM] [Info] Number of data points in the train set: 78104, number of used features: 25
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
LightGBM Results:
                precision    recall  f1-score   support

No Care Needed       0.78      0.82      0.80      9764
   Care Needed       0.59      0.54      0.57      4842

      accuracy                           0.73     14606
     macro avg       0.69      0.68      0.68     14606
  weighted avg       0.72      0.73      0.72     14606

ROC AUC Score: 0.7624


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [20]:
import pandas as pd

# Convert scaled arrays back to DataFrames to fix the warning
feature_names = X.columns.tolist()
X_train_sm_df = pd.DataFrame(X_train_sm, columns=feature_names)
X_test_df = pd.DataFrame(X_test_scaled, columns=feature_names)

# Tuned LightGBM
lgbm_model_v2 = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

lgbm_model_v2.fit(X_train_sm_df, y_train_sm)

y_pred_lgbm2 = lgbm_model_v2.predict(X_test_df)
y_pred_proba_lgbm2 = lgbm_model_v2.predict_proba(X_test_df)[:, 1]

print("LightGBM V2 (tuned):")
print(classification_report(y_test, y_pred_lgbm2, target_names=['No Care Needed', 'Care Needed']))
print(f"ROC AUC Score: {roc_auc_score(y_test, y_pred_proba_lgbm2):.4f}")

[LightGBM] [Info] Number of positive: 39052, number of negative: 39052
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003139 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2985
[LightGBM] [Info] Number of data points in the train set: 78104, number of used features: 25
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
LightGBM V2 (tuned):
                precision    recall  f1-score   support

No Care Needed       0.78      0.83      0.80      9764
   Care Needed       0.60      0.52      0.56      4842

      accuracy                           0.73     14606
     macro avg       0.69      0.67      0.68     14606
  weighted avg       0.72      0.73      0.72     14606

ROC AUC Score: 0.7600


In [21]:
# Load DBS biomarker dataset
df_bio, meta_bio = pyreadstat.read_dta('../data/4_LASI_W1_Biomarker.dta')

print(f"Biomarker dataset shape: {df_bio.shape}")
print(f"\nColumns: {df_bio.columns.tolist()}")

Biomarker dataset shape: (66859, 91)

Columns: ['prim_key', 'hhid', 'stateid', 'state', 'ssuid', 'residence', 'stateindividualweight', 'indiaindividualweight', 'bm001', 'bm002', 'bm003', 'bm004', 'bm006', 'bm007', 'bm008', 'bm010', 'bm011', 'bm012', 'bm014', 'bm015', 'bm016', 'bm017', 'bm018', 'bm019', 'bm020', 'bm021', 'bm022', 'bm024', 'bm025', 'bm026', 'bm028', 'bm029', 'bm030', 'bm031', 'bm032', 'bm033', 'bm034', 'bm036', 'bm037', 'bm038', 'bm039', 'bm040', 'bm041', 'bm042', 'bm043', 'bm044', 'bm045', 'bm046', 'bm047', 'bm048', 'bm049_iwer', 'bm050', 'bm051_iwer', 'bm052', 'bm053_iwer', 'bm054', 'bm055', 'bm056', 'bm057', 'bm058', 'bm058_other', 'bm059', 'bm060a', 'bm060b', 'bm061', 'bm062', 'bm063', 'bm064', 'bm065', 'bm066', 'bm067', 'bm068', 'bm069', 'bm071', 'bm072', 'bm073', 'bm074', 'bm076', 'bm077', 'bm079', 'bm080', 'bm080_other', 'bm080s1', 'bm080s2', 'bm080s3', 'bm080s4', 'bm080s5', 'bm080s6', 'bm080s7', 'bm081', 'interview_date']


In [22]:
# See what the biomarker columns actually mean
for col in df_bio.columns:
    label = meta_bio.column_names_to_labels.get(col, 'no label')
    missing = df_bio[col].isnull().sum()
    missing_pct = (missing / len(df_bio)) * 100
    if missing_pct < 50:  # only show columns with less than 50% missing
        print(f"{col}: {label} | Missing: {missing_pct:.1f}%")

prim_key: Primary key | Missing: 0.0%
hhid: Household identifier | Missing: 0.0%
stateid: Stateid | Missing: 0.0%
state: State | Missing: 0.0%
ssuid: Ssuid | Missing: 0.0%
residence: Place of residence | Missing: 0.0%
stateindividualweight: State individual weight | Missing: 0.0%
indiaindividualweight: India individual weight | Missing: 0.0%
bm001: Blood pressure consent | Missing: 0.1%
bm002: Have had smoke, alcohol, food or exercised 30 min. prior to the test | Missing: 0.2%
bm003: Local skin reactions where blood pressure cuff contact (left hand) | Missing: 0.2%
bm006: Blood pressure_1st systolic reading | Missing: 0.3%
bm007: Blood pressure_1st diastolic reading | Missing: 0.3%
bm008: Blood pressure_1st pulse reading | Missing: 0.3%
bm010: Blood pressure_2nd systolic reading | Missing: 0.3%
bm011: Blood pressure_2nd daistolic reading | Missing: 0.3%
bm012: Blood pressure_2nd pulse reading | Missing: 0.3%
bm014: Blood pressure_3rd systolic reading | Missing: 0.3%
bm015: Blood pressu

In [23]:
# Select best biomarker features
bio_features = [
    'prim_key',
    'bm017',  # avg systolic blood pressure
    'bm018',  # avg diastolic blood pressure
    'bm019',  # avg pulse
    'bm029',  # grip strength right hand
    'bm031',  # grip strength right hand 2nd measurement
    'bm056',  # timed walk 1st measurement (mobility)
    'bm057',  # timed walk 2nd measurement
    'bm067',  # height
    'bm071',  # weight
    'bm076',  # waist circumference
    'bm079',  # hip circumference
]

df_bio_selected = df_bio[bio_features].copy()

# Check missing
print("Missing values in biomarker features:")
print(df_bio_selected.isnull().sum())
print(f"\nBiomarker dataset rows: {len(df_bio_selected)}")

Missing values in biomarker features:
prim_key       0
bm017        179
bm018        179
bm019        179
bm029       1761
bm031       1763
bm056       1548
bm057       1548
bm067        749
bm071        756
bm076        781
bm079        779
dtype: int64

Biomarker dataset rows: 66859


In [24]:
# Rebuild ml_df with prim_key included
base_features = [
    'prim_key',  # add this for merging
    'dm003', 'dm005', 'ht001_a',
    'ht002', 'ht003', 'ht004', 'ht005', 'ht006',
    'ht007', 'ht008', 'ht009', 'ht010', 'ht301',
    'living_arrangements', 'dm006', 'we001',
    'fs601', 'ht014', 'ht015'
] + adl_cols

ml_df = df[base_features].copy()

# Recreate target variable
ml_df['needs_care'] = (ml_df[adl_cols] == 1).any(axis=1).astype(int)

# Drop ADL rows with missing
ml_df = ml_df.dropna(subset=adl_cols)

# Fill remaining missing with mode
for col in ml_df.columns:
    if ml_df[col].isnull().sum() > 0:
        ml_df[col] = ml_df[col].fillna(ml_df[col].mode()[0])

# Feature engineering
ml_df['disease_count'] = (ml_df[['ht002','ht003','ht004','ht005','ht006',
                                   'ht007','ht008','ht009','ht010']] == 1).sum(axis=1)
ml_df['age_group'] = pd.cut(ml_df['dm005'], bins=[0,60,70,80,120], labels=[0,1,2,3]).astype(int)
ml_df['lives_alone'] = (ml_df['living_arrangements'] == 1).astype(int)
ml_df['poor_health'] = (ml_df['ht001_a'] >= 4).astype(int)
ml_df['age_alone'] = ml_df['age_group'] * ml_df['lives_alone']
ml_df['disease_poor_health'] = ml_df['disease_count'] * ml_df['poor_health']

# Now merge biomarker
ml_df_bio = ml_df.merge(df_bio_selected, on='prim_key', how='left')

print(f"Shape after merge: {ml_df_bio.shape}")
print(f"Missing in biomarker cols:")
print(ml_df_bio[['bm017','bm018','bm019','bm029','bm031',
                   'bm056','bm057','bm067','bm071','bm076','bm079']].isnull().sum())

Shape after merge: (73027, 50)
Missing in biomarker cols:
bm017    6427
bm018    6427
bm019    6427
bm029    8008
bm031    8010
bm056    7796
bm057    7796
bm067    6997
bm071    7004
bm076    7029
bm079    7027
dtype: int64


In [25]:
bio_cols = ['bm017','bm018','bm019','bm029','bm031',
            'bm056','bm057','bm067','bm071','bm076','bm079']

# Fill missing biomarker values with median (these are continuous measurements)
for col in bio_cols:
    median_val = ml_df_bio[col].median()
    ml_df_bio[col] = ml_df_bio[col].fillna(median_val)

# Add BMI as engineered feature from height and weight
ml_df_bio['bmi'] = ml_df_bio['bm071'] / ((ml_df_bio['bm067'] / 100) ** 2)

# Average grip strength
ml_df_bio['avg_grip'] = (ml_df_bio['bm029'] + ml_df_bio['bm031']) / 2

# Average walk time
ml_df_bio['avg_walk'] = (ml_df_bio['bm056'] + ml_df_bio['bm057']) / 2

print("Missing values after filling:", ml_df_bio[bio_cols].isnull().sum().sum())
print(f"Dataset shape: {ml_df_bio.shape}")

# Now retrain with biomarker features
adl_cols = ['ht401','ht402','ht403','ht404','ht405','ht406',
            'ht407','ht408','ht409','ht410','ht411','ht412']

X = ml_df_bio.drop(columns=['needs_care', 'prim_key'] + adl_cols)
y = ml_df_bio['needs_care']

print(f"\nTotal features: {X.shape[1]}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train_scaled, y_train)

# Train LightGBM
X_train_sm_df = pd.DataFrame(X_train_sm, columns=X.columns.tolist())
X_test_df = pd.DataFrame(X_test_scaled, columns=X.columns.tolist())

lgbm_bio = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

lgbm_bio.fit(X_train_sm_df, y_train_sm)

y_pred_bio = lgbm_bio.predict(X_test_df)
y_pred_proba_bio = lgbm_bio.predict_proba(X_test_df)[:, 1]

print("\nLightGBM + Biomarker Features:")
print(classification_report(y_test, y_pred_bio, target_names=['No Care Needed', 'Care Needed']))
print(f"ROC AUC Score: {roc_auc_score(y_test, y_pred_proba_bio):.4f}")

Missing values after filling: 0
Dataset shape: (73027, 53)

Total features: 39
[LightGBM] [Info] Number of positive: 39052, number of negative: 39052
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004810 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6930
[LightGBM] [Info] Number of data points in the train set: 78104, number of used features: 39
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000

LightGBM + Biomarker Features:
                precision    recall  f1-score   support

No Care Needed       0.77      0.86      0.81      9764
   Care Needed       0.63      0.50      0.56      4842

      accuracy                           0.74     14606
     macro avg       0.70      0.68      0.68     14606
  weighted avg       0.73      0.74      0.73     14606

ROC AUC Score: 0.7698


In [26]:
# Feature importance for biomarker model
feature_names = X.columns.tolist()
importances = lgbm_bio.feature_importances_
indices = np.argsort(importances)[::-1]

print("Top 20 most important features:")
for i in range(20):
    print(f"{i+1}. {feature_names[indices[i]]}: {importances[indices[i]]}")
    

Top 20 most important features:
1. dm005: 2960
2. bm019: 1947
3. bm067: 1738
4. bm017: 1660
5. bm018: 1643
6. bm079: 1628
7. bm056: 1610
8. bm076: 1502
9. bm029: 1477
10. bmi: 1475
11. bm031: 1394
12. bm071: 1331
13. ht001_a: 1279
14. bm057: 1262
15. avg_grip: 1145
16. avg_walk: 1068
17. living_arrangements: 939
18. age_group: 607
19. ht015: 445
20. ht008: 427


In [28]:
# Convert all columns to numeric
X_train_df = X_train_df.apply(pd.to_numeric, errors='coerce').fillna(0)
X_test_df = X_test_df.apply(pd.to_numeric, errors='coerce').fillna(0)

# Now train
lgbm_bio_v2 = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.02,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale,
    random_state=42,
    n_jobs=-1
)

lgbm_bio_v2.fit(
    X_train_df, y_train,
    eval_set=[(X_test_df, y_test)],
    callbacks=[lgbm.early_stopping(50, verbose=False)]
)

y_pred_bio2 = lgbm_bio_v2.predict(X_test_df)
y_pred_proba_bio2 = lgbm_bio_v2.predict_proba(X_test_df)[:, 1]

print("\nLightGBM + Biomarkers + Native Imbalance Handling:")
print(classification_report(y_test, y_pred_bio2, target_names=['No Care Needed', 'Care Needed']))
print(f"ROC AUC Score: {roc_auc_score(y_test, y_pred_proba_bio2):.4f}")

[LightGBM] [Info] Number of positive: 19369, number of negative: 39052
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002947 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3230
[LightGBM] [Info] Number of data points in the train set: 58421, number of used features: 39
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.331542 -> initscore=-0.701220
[LightGBM] [Info] Start training from score -0.701220

LightGBM + Biomarkers + Native Imbalance Handling:
                precision    recall  f1-score   support

No Care Needed       0.82      0.73      0.77      9764
   Care Needed       0.55      0.67      0.60      4842

      accuracy                           0.71     14606
     macro avg       0.68      0.70      0.69     14606
  weighted avg       0.73      0.71      0.71     14606

ROC AUC Score: 0.7726


In [29]:
# Check cognitive composite scores
cog_cols = ['mh021a', 'mh022', 'mh036', 'mh037', 'mh038_1', 'mh038_2']

for col in cog_cols:
    if col in df.columns:
        label = meta.column_names_to_labels.get(col, 'no label')
        missing_pct = df[col].isnull().sum() / len(df) * 100
        print(f"✅ {col}: {label}")
        print(f"   Missing: {missing_pct:.1f}%")
        print(f"   Values: {df[col].value_counts().head(3).to_dict()}")
        print()
    else:
        print(f"❌ {col}: NOT FOUND")

✅ mh021a: Number series_verbal consent
   Missing: 3.2%
   Values: {2: 42595, 1: 28441}

✅ mh022: Number series_missing value-block 7
   Missing: 61.6%
   Values: {10: 14123, 9: 8793, 11: 2546}

✅ mh036: Numeric ability_aptness to count backward
   Missing: 2.4%
   Values: {1: 36177, 3: 21809, 2: 13676}

✅ mh037: Numeric ability_time taken to count backward beginning with 20
   Missing: 32.1%
   Values: {13: 2158, 12: 2156, 15: 2041}

✅ mh038_1: Numeric ability_R counted correctly up to
   Missing: 72.4%
   Values: {1: 5341, 90: 1134, 100: 898}

✅ mh038_2: Numeric ability_R counted incorrectly but counted correctly up to
   Missing: 84.3%
   Values: {90: 981, 89: 878, 80: 709}



In [30]:
# Add cognitive features
cog_cols = ['prim_key', 'mh021a', 'mh036']
df_cog = df[cog_cols].copy()

# Merge into ml_df_bio
ml_df_bio = ml_df_bio.merge(df_cog, on='prim_key', how='left')

# Fill missing with mode
for col in ['mh021a', 'mh036']:
    ml_df_bio[col] = ml_df_bio[col].fillna(ml_df_bio[col].mode()[0])

print(f"Shape after adding cognitive features: {ml_df_bio.shape}")
print(f"Missing: {ml_df_bio[['mh021a','mh036']].isnull().sum().to_dict()}")

# Retrain
X = ml_df_bio.drop(columns=['needs_care', 'prim_key'] + adl_cols)
y = ml_df_bio['needs_care']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos

X_train_df = pd.DataFrame(X_train, columns=X.columns.tolist())
X_train_df = X_train_df.apply(pd.to_numeric, errors='coerce').fillna(0)

X_test_df = pd.DataFrame(X_test, columns=X.columns.tolist())
X_test_df = X_test_df.apply(pd.to_numeric, errors='coerce').fillna(0)

lgbm_final = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.02,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale,
    random_state=42,
    n_jobs=-1
)

lgbm_final.fit(
    X_train_df, y_train,
    eval_set=[(X_test_df, y_test)],
    callbacks=[lgbm.early_stopping(50, verbose=False)]
)

y_pred_final = lgbm_final.predict(X_test_df)
y_pred_proba_final = lgbm_final.predict_proba(X_test_df)[:, 1]

print("\nLightGBM Final (+ Cognitive Features):")
print(classification_report(y_test, y_pred_final, target_names=['No Care Needed', 'Care Needed']))
print(f"ROC AUC Score: {roc_auc_score(y_test, y_pred_proba_final):.4f}")

Shape after adding cognitive features: (73027, 55)
Missing: {'mh021a': 0, 'mh036': 0}
[LightGBM] [Info] Number of positive: 19369, number of negative: 39052
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003231 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3237
[LightGBM] [Info] Number of data points in the train set: 58421, number of used features: 41
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.331542 -> initscore=-0.701220
[LightGBM] [Info] Start training from score -0.701220

LightGBM Final (+ Cognitive Features):
                precision    recall  f1-score   support

No Care Needed       0.82      0.73      0.77      9764
   Care Needed       0.55      0.67      0.60      4842

      accuracy                           0.71     14606
     macro avg       0.68      0.70      0.69     14606
  weighted avg       0.73      0.7